# Reproducibility Pipeline: BIRD Extraction, L0-L4 Layer Classification, Schema-Case Revision

This notebook reproduces the data-preparation pipeline used by the metadata-ablation study. It has three sections:

1. **BIRD dataset extraction**: download BIRD mini-dev and verify the structure.
2. **L0-L4 layer classification**: generate the four metadata documents per database and map them into the five cumulative levels (L0 to L4).
3. **Schema-case revision**: rewrite identifier casing in the generated docs to match the live PostgreSQL schema (the documented fix for Type-A case-mismatch failures).

Sections 1 and 2 require no external services. Section 3 requires a live `bird_minidev` PostgreSQL database loaded from BIRD's official `BIRD_dev.sql` dump; if that database is not available, the schema-revision cell stays documented but unexecuted.

Runs on Google Colab or locally with `pip install -r ../requirements.txt`.

## 1. BIRD dataset extraction

BIRD mini-dev is the PostgreSQL slice of the BIRD development set: 11 databases, ~500 human-annotated question-and-gold-SQL pairs.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WORK_DIR = REPO_ROOT / 'bird_mini_dev'
ZIP_PATH = WORK_DIR / 'minidev.zip'
DATA_DIR = WORK_DIR / 'minidev' / 'MINIDEV'
OUTPUT_DIR = REPO_ROOT / 'content-out' / 'bird-docs'

WORK_DIR.mkdir(exist_ok=True)

if not DATA_DIR.exists():
    print('Downloading BIRD mini-dev (~800MB)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    subprocess.check_call(['gdown', 'https://drive.google.com/uc?id=13VLWIwpw5E3d5DUkMvzw7hvHE67a4XkG', '-O', str(ZIP_PATH)])
    subprocess.check_call(['unzip', '-q', '-o', str(ZIP_PATH), '-d', str(WORK_DIR)])
    os.remove(ZIP_PATH)
    print(f'Done. Data at: {DATA_DIR}')
else:
    print(f'Data already present at: {DATA_DIR}')

Data already present at: /Users/gary/myapps/4d-metadata-ablation/bird_mini_dev/minidev/MINIDEV


In [2]:
DEV_TABLES_JSON = DATA_DIR / 'dev_tables.json'
QUESTIONS_JSON = DATA_DIR / 'mini_dev_postgresql.json'
DEV_DATABASES_DIR = DATA_DIR / 'dev_databases'

assert DEV_TABLES_JSON.exists(), f'Missing: {DEV_TABLES_JSON}'
assert QUESTIONS_JSON.exists(), f'Missing: {QUESTIONS_JSON}'
assert DEV_DATABASES_DIR.exists(), f'Missing: {DEV_DATABASES_DIR}'

databases = sorted(p.name for p in DEV_DATABASES_DIR.iterdir() if p.is_dir())
print(f'Found {len(databases)} databases:')
for d in databases:
    print(f'  {d}')

Found 11 databases:
  california_schools
  card_games
  codebase_community
  debit_card_specializing
  european_football_2
  financial
  formula_1
  student_club
  superhero
  thrombosis_prediction
  toxicology


## 2. L0-L4 layer classification

The bird-loader pipeline generates four markdown documents per database, one per metadata dimension. The five cumulative levels (L0 to L4) and the four leave-one-out conditions (L4-DD, L4-QP, L4-BC, L4-DK) are then constructed at experiment time by selecting which of the four documents to inject into the model prompt.

| Level | Dimensions present |
|---|---|
| L0    | (none; bare schema only) |
| L1    | data_dictionary |
| L2    | data_dictionary, query_patterns |
| L3    | data_dictionary, query_patterns, business_context |
| L4    | data_dictionary, query_patterns, business_context, domain_knowledge |
| L4-DD | query_patterns, business_context, domain_knowledge (DD dropped) |
| L4-QP | data_dictionary, business_context, domain_knowledge (QP dropped) |
| L4-BC | data_dictionary, query_patterns, domain_knowledge (BC dropped) |
| L4-DK | data_dictionary, query_patterns, business_context (DK dropped) |

The L0-PAD condition is constructed separately by replacing the L4 metadata budget with synthetic non-BIRD filler text to keep the prompt-token count constant; it is not produced by the bird-loader pipeline.

In [3]:
import sys
sys.path.insert(0, str(REPO_ROOT))

# Point the bird_loader config at the downloaded data so its module-level paths resolve correctly.
import bird_loader.config as bl_config
bl_config.MINIDEV_DIR = DATA_DIR
bl_config.DEV_TABLES_JSON = DEV_TABLES_JSON
bl_config.QUESTIONS_JSON = QUESTIONS_JSON
bl_config.DEV_DATABASES_DIR = DEV_DATABASES_DIR
bl_config.OUTPUT_DIR = OUTPUT_DIR

from bird_loader.loaders import load_tables, load_questions, load_csv_descriptions
from bird_loader import generators

tables = load_tables()
questions = load_questions()
print(f'Loaded {len(tables)} databases, {sum(len(v) for v in questions.values())} questions.')

Loaded 11 databases, 500 questions.


In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENERATORS = {
    'DataDictionary.md': generators.data_dictionary,
    'QueryPatterns.md':  generators.query_patterns,
    'BusinessContext.md': generators.business_context,
    'DomainContext.md':  generators.domain_context,
}

total_files = 0
for db_name, db in tables.items():
    db_out = OUTPUT_DIR / db_name
    db_out.mkdir(parents=True, exist_ok=True)
    csv_descs = load_csv_descriptions(db_name)
    db_questions = questions.get(db_name, [])
    for filename, gen_func in GENERATORS.items():
        (db_out / filename).write_text(gen_func(db, db_questions, csv_descs))
        total_files += 1
    print(f'  {db_name}: {len(db_questions)} questions -> {db_out}')

print(f'\nGenerated {total_files} files in {OUTPUT_DIR}/')

  debit_card_specializing: 30 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/debit_card_specializing
  financial: 32 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/financial
  formula_1: 66 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/formula_1
  california_schools: 30 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/california_schools
  card_games: 52 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/card_games
  european_football_2: 51 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/european_football_2
  thrombosis_prediction: 50 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/thrombosis_prediction
  toxicology: 40 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/toxicology
  student_club: 48 questions -> /Users/gary/myapps/4d-metadata-ablation/content-out/bird-docs/stud

In [5]:
# Preview a sample of the generated docs for one database.
from IPython.display import Markdown, display

sample_db = 'thrombosis_prediction'
for doc in GENERATORS:
    path = OUTPUT_DIR / sample_db / doc
    display(Markdown(f'---\n### {doc} ({sample_db})'))
    preview = '\n'.join(path.read_text().splitlines()[:30])
    display(Markdown(preview))

---
### DataDictionary.md (thrombosis_prediction)

# Data Dictionary: thrombosis_prediction


## Examination

| Column | Human-Readable Name | Data Type | Description | Value Notes |
|--------|-------------------|-----------|-------------|-------------|
| ID |  | integer | identification of the patient |  |
| Examination Date |  | date | Examination Date |  |
| aCL IgG | anti-Cardiolipin antibody (IgG) | real | anti-Cardiolipin antibody (IgG) concentration |  |
| aCL IgM | anti-Cardiolipin antibody (IgM) | real | anti-Cardiolipin antibody (IgM) concentration |  |
| ANA | anti-nucleus antibody | integer | anti-nucleus antibody concentration |  |
| ANA Pattern | pattern observed in the sheet of ANA examination | text | pattern observed in the sheet of ANA examination |  |
| aCL IgA | anti-Cardiolipin antibody (IgA) concentration | integer | anti-Cardiolipin antibody (IgA) concentration |  |
| Diagnosis |  | text | disease names |  |
| KCT | measure of degree of coagulation | text | measure of degree of coagulation | +: positive  -: negative |
| RVVT | measure of degree of coagulation | text | measure of degree of coagulation | +: positive  -: negative |
| LAC | measure of degree of coagulation | text | measure of degree of coagulation | +: positive  -: negative |
| Symptoms |  | text | other symptoms observed |  |
| Thrombosis |  | integer | degree of thrombosis | 0: negative (no thrombosis) 1: positive (the most severe one) 2: positive (severe)3: positive (mild) |

## Patient

| Column | Human-Readable Name | Data Type | Description | Value Notes |
|--------|-------------------|-----------|-------------|-------------|
| ID |  | integer | identification of the patient |  |
| SEX |  | text | Sex | F: female; M: male |
| Birthday |  | date | Birthday |  |
| Description |  | date | the first date when a patient data was recorded | null or empty: not recorded |
| First Date |  | date | the date when a patient came to the hospital |  |

---
### QueryPatterns.md (thrombosis_prediction)

# Query Patterns: thrombosis_prediction

## Primary Keys

| Table | Primary Key Column(s) |
|-------|----------------------|
| Patient | ID |
| Laboratory | ID, Date |

## Foreign Key Relationships

### Examination → Patient

- **FK Column:** `Examination.ID`
- **References:** `Patient.ID`

```sql
SELECT *
FROM Examination
INNER JOIN Patient
  ON Examination.ID = Patient.ID;
```

### Laboratory → Patient

- **FK Column:** `Laboratory.ID`
- **References:** `Patient.ID`

```sql
SELECT *

---
### BusinessContext.md (thrombosis_prediction)

# Business Context: thrombosis_prediction

## Column Synonyms

Mappings between original column names and human-readable names.

| Original Column | Table | Human-Readable Name |
|----------------|-------|-------------------|
| aCL IgG | Examination | anti-Cardiolipin antibody (IgG) |
| aCL IgM | Examination | anti-Cardiolipin antibody (IgM) |
| ANA | Examination | anti-nucleus antibody |
| ANA Pattern | Examination | pattern observed in the sheet of ANA examination |
| aCL IgA | Examination | anti-Cardiolipin antibody (IgA) concentration |
| KCT | Examination | measure of degree of coagulation |
| RVVT | Examination | measure of degree of coagulation |
| LAC | Examination | measure of degree of coagulation |
| GOT | Laboratory | AST glutamic oxaloacetic transaminase |
| GPT | Laboratory | ALT glutamic pyruvic transaminase |
| LDH | Laboratory | lactate dehydrogenase |
| ALP | Laboratory | alkaliphophatase |
| TP | Laboratory | total protein |
| ALB | Laboratory | albumin |
| UA | Laboratory | uric acid |
| UN | Laboratory | urea nitrogen |
| CRE | Laboratory | creatinine |
| T-BIL | Laboratory | total bilirubin |
| T-CHO | Laboratory | total cholesterol |
| TG | Laboratory | triglyceride |
| CPK | Laboratory | creatinine phosphokinase |
| GLU | Laboratory | blood glucose |

---
### DomainContext.md (thrombosis_prediction)

# Domain Context: thrombosis_prediction

## KPIs & Calculations

- **in-patient refers to Admission = '+'**
  - _From:_ Are there more in-patient or outpatient who were male? What is the deviation in percentage?

- **outpatient refers to Admission = '-'**
  - _From:_ Are there more in-patient or outpatient who were male? What is the deviation in percentage?

- **percentage = DIVIDE(COUNT(ID) where SEX = 'M' and Admission = '+', COUNT(ID) where SEX  = 'M' and Admission = '-')**
  - _From:_ Are there more in-patient or outpatient who were male? What is the deviation in percentage?

- **calculation = DIVIDE(COUNT(ID) where year(Birthday) > '1930' and SEX = 'F'), (COUNT(ID) where SEX = 'F')**
  - _From:_ What is the percentage of female patient were born after 1930?

- **inpatient refers to Admission = '+'**
  - _From:_ What is the ratio of outpatient to inpatient followed up treatment among all the 'SLE' diagnosed patient?

- **calculation =  DIVIDE(COUNT(ID) where Diagnosis = 'SLE' and Admission = '+', COUNT(ID) where Diagnosis = 'SLE' and Admission = '-')**
  - _From:_ What is the ratio of outpatient to inpatient followed up treatment among all the 'SLE' diagnosed patient?

- **positive degree of coagulation refers to RVVT = '+'**
  - _From:_ State the ID and age of patient with positive degree of coagulation.

- **immediately followed at the outpatient clinic refers to Admission = '-'**
  - _From:_ How many female patients who came at the hospital in 1997 was immediately followed at the outpatient clinic?

- **calculation = DIVIDE(SUM(UA <= '8.0' and SEX = 'M'), SUM(UA <= '6.5 and SEX = 'F'))**
  - _From:_ What is the ratio of male to female patients among all those with abnormal uric acid counts?

## 3. Schema-case revision (the revised metadata additions)

BIRD's source CSV descriptions sometimes refer to tables and columns in casing that differs from the live PostgreSQL schema (for example `Patient` in the docs vs `patient` in the database, or vice versa). PostgreSQL is case-sensitive on quoted identifiers, so this mismatch causes a Type-A failure mode (the model generates syntactically valid SQL that the database rejects). The thrombosis_prediction and european_football_2 hostile-database patterns reported in the paper are exactly this failure mode.

The schema-case revision tool (`scripts/revise_docs_against_schema.py`) reads the generated docs in `content-out/bird-docs/`, introspects the live database schema via `information_schema`, and rewrites identifier casing where the docs disagree with the database. It is conservative: only compound identifiers (underscored or camel-case tokens) are rewritten, and the tool validates that no non-identifier byte changes.

This cell requires a running `bird_minidev` PostgreSQL database loaded from BIRD's `BIRD_dev.sql` dump. If that database is not available locally, leave the cell unexecuted; the cell below documents the expected invocation.

In [6]:
# Uncomment after loading the bird_minidev PostgreSQL database locally.
#
# import subprocess, sys
# subprocess.check_call([
#     sys.executable, str(REPO_ROOT / 'scripts' / 'revise_docs_against_schema.py'),
#     '--input-dir',  str(REPO_ROOT / 'content-out' / 'bird-docs'),
#     '--output-dir', str(REPO_ROOT / 'content-out' / 'bird-docs-revised'),
#     '--database-url', 'postgresql+psycopg://localhost/bird_minidev',
# ])
print('Schema-case revision cell is documented but not executed by default.')
print('Provide a live bird_minidev PostgreSQL database and uncomment the call above to run it.')

Schema-case revision cell is documented but not executed by default.
Provide a live bird_minidev PostgreSQL database and uncomment the call above to run it.


## 4. From metadata documents to the experiment

Once `content-out/bird-docs-revised/` is populated, the experiment runner consumes those revised documents at evaluation time. It parses the four per-dimension markdown files, injects the appropriate dimension subset for each level into the model prompt, and writes one per-cell JSON file per (database, level, model, repetition) combination. The model outputs of that runner ship in this repo under `results/{9b,27b,30b,gemma4,qwen36,opus47}/`.

Notebook 02 (`02-analysis-and-charts.ipynb`) starts from those per-cell JSON files and reproduces every table and figure in the paper.